# Nemotron Model Reasoning Challenge — GSPO (RLVR) stage

This notebook runs the **RLVR / GSPO** stage on the **RTX PRO 6000 (96 GB)** GPU, **warm-started from the SFT LoRA adapter**. It is the **highest-ceiling lever** in the pipeline (it optimises directly against the verifiable reward) but is **compute-heavy** (G× rollouts on a 30B MoE per step).

The base model is an **MoE** (Mamba‑MoE), so we use **GSPO** — *sequence-level* importance sampling (`importance_sampling_level="sequence"`), which is MoE-stable — **not** token-level GRPO. The reward is **verifiable boxed-match** (no reward model): an answer scores 1.0 when the string inside `\boxed{}` matches the gold answer exactly or numerically within ±1e-2.

> **UNTESTED ON GPU — this is a first GSPO draft; expect to tune** (NUM_GENERATIONS / MAX_PROMPTS / MAX_STEPS / LR / BETA, and the `use_vllm` flag).

## Required Kaggle UI setup (do this before running)

In the notebook editor sidebar:

1. **Accelerator** → `GPU RTX PRO 6000`
2. **Internet** → `ON`  *(this kernel pip-installs a recent `trl` + `vllm` — see the install cell)*
3. **Add Input** → attach all four:
   - the **competition** dataset (provides `train.csv`)
   - the model **`metric/nemotron-3-nano-30b-a3b-bf16`**
   - the utility script **`ryanholbrook/nvidia-utility-script`** (provides `nvidia_cutlass_dsl`, required for `mamba_ssm` to import)
   - the **SFT LoRA adapter to warm-start from** — attach the train kernel's output (`kaggle_train_submit.ipynb`) **or** an adapter dataset, so that `adapter_config.json` + `adapter_model.safetensors` land under `/kaggle/input/`. We warm-start from the **0.61 SFT adapter**.

## Answer format the grader expects

```
<think>...</think>\boxed{answer}
```

LoRA **rank must be <= 32**. The graded answer is whatever is inside `\boxed{}`.

## Config knobs

GSPO is expensive (`NUM_GENERATIONS` rollouts per prompt on a 30B MoE). Keep `MAX_PROMPTS` / `MAX_STEPS` **small first** to confirm the pipeline runs end-to-end, then scale up.

In [ ]:
NUM_GENERATIONS = 8  # G rollouts per prompt (group)
MAX_PROMPTS = 512  # subset of train prompts for RL (keep small first)
MAX_STEPS = 100  # GSPO optimizer steps
MAX_COMPLETION_LEN = 1024
LR = 1e-6
BETA = 0.04  # KL to the SFT reference (anti-collapse)
TEMPERATURE = 1.0
SEED = 42
HOLDOUT_N = 200  # exclude same seeded holdout as eval notebook

print(
    f"NUM_GENERATIONS={NUM_GENERATIONS} | MAX_PROMPTS={MAX_PROMPTS} | "
    f"MAX_STEPS={MAX_STEPS} | MAX_COMPLETION_LEN={MAX_COMPLETION_LEN} | "
    f"LR={LR} | BETA={BETA} | TEMPERATURE={TEMPERATURE} | SEED={SEED} | "
    f"HOLDOUT_N={HOLDOUT_N}"
)

In [ ]:
# Internet must be ON. GSPO (importance_sampling_level="sequence") needs a recent trl;
# the BYOD image's trl may be too old, so we pin trl>=0.21. vllm accelerates rollouts.
# Leave transformers as the image's version — the custom Nemotron code depends on it; do NOT -U transformers.
!pip install -q -U "trl>=0.21" vllm

In [ ]:
# Load train.csv robustly (glob so the attached dataset's folder name doesn't matter),
# then build raw id/prompt/answer lists. The chat-templated Dataset is built AFTER the
# setup cell, because `tokenizer` isn't defined until then.
import glob
import random

candidates = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True))
assert candidates, (
    "train.csv not found under /kaggle/input - attach the competition dataset."
)
TRAIN_CSV = candidates[0]
print("Using:", TRAIN_CSV)

try:
    import polars as pl

    train = pl.read_csv(TRAIN_CSV)
    ids = [str(x) for x in train["id"].to_list()]
    prompts = train["prompt"].to_list()
    answers = [str(a) for a in train["answer"].to_list()]
except Exception as e:
    print("polars unavailable, falling back to pandas:", e)
    import pandas as pd

    train = pd.read_csv(TRAIN_CSV)
    ids = [str(x) for x in train["id"].tolist()]
    prompts = train["prompt"].tolist()
    answers = [str(a) for a in train["answer"].tolist()]

print("rows:", len(ids))

# Reserve the SAME seeded holdout as the eval notebook (drop the last HOLDOUT_N of the
# seeded shuffle) so RL never trains on eval rows.
if HOLDOUT_N > 0 and len(ids) > HOLDOUT_N:
    _order = list(range(len(ids)))
    random.Random(SEED).shuffle(_order)
    _hold = set(_order[-HOLDOUT_N:])
    _keep = [j for j in range(len(ids)) if j not in _hold]
    ids = [ids[j] for j in _keep]
    prompts = [prompts[j] for j in _keep]
    answers = [answers[j] for j in _keep]
    print(f"excluded {len(_hold)} holdout rows; {len(ids)} remain")

# Take the first MAX_PROMPTS for RL.
prompts = prompts[:MAX_PROMPTS]
answers = answers[:MAX_PROMPTS]
ids = ids[:MAX_PROMPTS]
print(f"using {len(prompts)} prompts for RL")

In [ ]:
# Setup: vendored cutlass + executable Triton ptxas (Blackwell) BEFORE importing mamba_ssm.
import glob
import os
import shutil
import site

# 1. cutlass (auto-discover; mount uses underscores: nvidia_utility_script)
cands = sorted(
    set(
        glob.glob("/kaggle/usr/lib/**/python_packages", recursive=True)
        + glob.glob("/kaggle/input/**/python_packages", recursive=True)
    )
)
print("python_packages dirs:", cands)
for c in cands:
    site.addsitedir(c)
for h in glob.glob(
    "/kaggle/**/nvidia_cutlass_dsl/python_packages/cutlass/__init__.py", recursive=True
):
    site.addsitedir(os.path.dirname(os.path.dirname(h)))

# 2. Copy vendored Triton nvidia bin to a writable+exec dir and point Triton at it via env
#    (the env override is read by the vendored Triton's knobs; verified working).
ptxas_hits = glob.glob(
    "/kaggle/usr/lib/**/triton/backends/nvidia/bin/ptxas-blackwell", recursive=True
)
print("ptxas-blackwell candidates:", ptxas_hits)
if ptxas_hits:
    srcbin = os.path.dirname(ptxas_hits[0])
    dstbin = "/tmp/triton_nvidia_bin"
    os.makedirs(dstbin, exist_ok=True)
    for b in glob.glob(os.path.join(srcbin, "*")):
        d = os.path.join(dstbin, os.path.basename(b))
        try:
            shutil.copy(b, d)
            os.chmod(d, 0o755)
        except OSError as e:
            print("copy skip:", b, e)
    wp = os.path.join(dstbin, "ptxas-blackwell")
    for var in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH"):
        os.environ[var] = wp
    os.environ["PATH"] = dstbin + ":" + os.environ.get("PATH", "")
    print("ptxas (writable):", wp, "executable:", os.access(wp, os.X_OK))

import kagglehub
import mamba_ssm  # noqa: F401  (registers Mamba CUDA kernels)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 3. The vendored Triton's NvidiaTool dataclass is unhashable (eq=True, no __hash__) and gets
#    hashed during Blackwell kernel compilation -> add a __hash__ so backward() can compile.
try:
    from triton import knobs

    if getattr(knobs.NvidiaTool, "__hash__", None) is None:
        knobs.NvidiaTool.__hash__ = lambda self: hash(getattr(self, "path", id(self)))
        print("patched NvidiaTool.__hash__")
    print("ptxas_blackwell knob:", knobs.nvidia.ptxas_blackwell.path)
except Exception as ex:
    print("triton knob inspect failed:", repr(ex))

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print("MODEL_PATH:", MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded")

In [ ]:
# Build the RL dataset (needs `tokenizer`). Each `prompt` is the chat-templated USER turn
# WITH the generation prompt appended, so the model continues from the assistant turn.
from datasets import Dataset

ds = Dataset.from_list(
    [
        {
            "prompt": tokenizer.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False,
                add_generation_prompt=True,
            ),
            "answer": a,
        }
        for p, a in zip(prompts, answers, strict=False)
    ]
)
print(ds)
print("=== example prompt ===")
print(ds[0]["prompt"])
print("=== example answer ===", ds[0]["answer"])

In [ ]:
# Warm-start from the attached SFT LoRA adapter (is_trainable=True so GSPO updates it).
# If no adapter is attached, fall back to base — RL from base is weaker but still runs.
import glob
import os

from peft import PeftModel

adapter_cfgs = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
if adapter_cfgs:
    adapter_dir = os.path.dirname(adapter_cfgs[0])
    print("warm-starting from adapter:", adapter_dir)
    model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=True)
    model.print_trainable_parameters()
else:
    print(
        "WARNING: no adapter_config.json found under /kaggle/input - "
        "attach the SFT adapter to warm-start. Continuing from BASE (weaker RL)."
    )

In [ ]:
# Verifiable rewards (mirror src/train/rl.py, the tested source of truth). TRL reward
# signature: fn(completions, **dataset_cols) -> list[float]; `answer` arrives as a kwarg.
import re


def _extract_boxed(text):
    marker = "\\boxed{"
    s = text.rfind(marker)
    if s == -1:
        return None
    i = s + len(marker)
    depth = 1
    out = []
    while i < len(text) and depth > 0:
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(ch)
        i += 1
    return "".join(out)


def _score(pred, gold, tol=1e-2):
    if pred is None:
        return False
    p, g = pred.strip(), str(gold).strip()
    if p == g:
        return True
    try:
        return abs(float(p) - float(g)) <= tol
    except ValueError:
        return False


def _text(c):
    return (
        c
        if isinstance(c, str)
        else (c[-1].get("content", "") if isinstance(c, list) and c else str(c))
    )


def boxed_reward(completions, answer, **kw):
    return [
        1.0 if _score(_extract_boxed(_text(c)), a) else 0.0
        for c, a in zip(completions, answer, strict=False)
    ]


_fmt = re.compile(r"<think>.*?</think>.*?\\boxed\{.*?\}", re.I | re.S)


def format_reward(completions, **kw):
    return [0.2 if _fmt.search(_text(c)) else 0.0 for c in completions]


# sanity check on a tiny synthetic batch
_demo = ["<think>x</think>\\boxed{42}", "\\boxed{7}", "no answer"]
print("boxed_reward:", boxed_reward(_demo, ["42", "99", "3"]))
print("format_reward:", format_reward(_demo))

## GSPO training

The only difference from GRPO is `importance_sampling_level="sequence"` — sequence-level importance sampling, which keeps the MoE stable. Two failure modes to watch for:

- If `importance_sampling_level` is **rejected** by `GRPOConfig`, the installed `trl` is too old — bump the pin in the pip-install cell (`trl>=0.21`) and restart.
- If `use_vllm=True` **errors** on the custom Nemotron model (vLLM may not support its custom code), set `use_vllm=False` to fall back to slower HF-generate rollouts.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

cfg = GRPOConfig(
    output_dir="/kaggle/working/gspo",
    importance_sampling_level="sequence",  # <-- GSPO (vs "token" = GRPO)
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LEN,
    temperature=TEMPERATURE,
    learning_rate=LR,
    beta=BETA,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=NUM_GENERATIONS,  # one prompt's group per step is fine
    gradient_accumulation_steps=1,
    logging_steps=1,
    save_strategy="no",
    report_to=[],
    use_vllm=True,  # fast rollouts; if it errors, set False
    seed=SEED,
)
trainer = GRPOTrainer(
    model=model,
    args=cfg,
    reward_funcs=[boxed_reward, format_reward],
    train_dataset=ds,
)
trainer.train()
trainer.save_model("/kaggle/working/gspo")

## Package the submission

`GRPOTrainer.save_model` writes the (warm-started, now RL-tuned) adapter to `/kaggle/working/gspo`. We zip **only** the adapter files into `submission.zip` (named explicitly so nothing else gets swept in).

In [ ]:
import os
import subprocess

# GRPOTrainer.save_model writes the adapter to /kaggle/working/gspo
src = "/kaggle/working/gspo"
os.chdir(src)
subprocess.run(
    "zip -m /kaggle/working/submission.zip adapter_config.json adapter_model.safetensors",
    shell=True,
    check=True,
)
print("submission.zip ready:", os.path.exists("/kaggle/working/submission.zip"))

## Submit + caveats

Submit `/kaggle/working/submission.zip` to the competition. **Validate first** with the holdout-eval kernel (`kaggle_eval_holdout.ipynb`) — it scores against the same seeded `HOLDOUT_N` rows excluded here, so you can confirm GSPO actually moved accuracy before spending a submission.

```bash
kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f submission.zip \
  -m "GSPO RLVR warm-started from 0.61 SFT adapter"
```

**Caveats / tuning:**

- GSPO is **compute-heavy** — `NUM_GENERATIONS`× rollouts per prompt on a 30B MoE. Keep `MAX_PROMPTS` / `MAX_STEPS` small initially, confirm reward trends up, then scale.
- We **warm-start from the 0.61 SFT adapter** — RL refines an already-competent policy; starting from base is weaker.
- `BETA` (KL to the SFT reference) guards against reward-hacking / collapse — raise it if completions degenerate, lower it if learning stalls.
- If `importance_sampling_level` is rejected → trl too old (bump the pin). If `use_vllm=True` errors on the custom model → set it `False`.
- Adapter must keep LoRA **rank <= 32** (inherited from the warm-start adapter); answers graded inside `\boxed{}` after a `<think>...</think>` block.